In [1]:
# mike babb
# created: 2026 08 23
# find five groups of five letters

In [2]:
# standard
import pickle
import os
import sqlite3
from string import ascii_lowercase
from itertools import product, permutations, combinations

In [3]:
import pandas as pd
import numpy as np
import networkx as nx

In [4]:
# custom
from utils import *

# LOAD LETTERS

In [5]:
letter_dict = load_pickle(file_name = 'letter_dict.pkl')

In [6]:
word_df = pd.read_csv(filepath_or_buffer=  'words_alpha.txt', header = None, names = ['word'], dtype = str)

In [7]:
word_df['word'] = word_df['word'].astype(str)

In [8]:
word_df.head()

,word
0,a
1,aa
2,aaa
3,aah
4,aahed


In [9]:
word_df.shape

(370105, 1)

In [10]:
word_df['word'].isna().value_counts()

word
False    370105
Name: count, dtype: int64

In [11]:
word_df['lcase'] = word_df['word'].str.lower()

In [12]:
word_df['n_letters'] = word_df['word'].str.len()

In [13]:
word_df['letters_sorted'] = word_df['lcase'].map(lambda x: ''.join(sorted(x)))

In [14]:
word_df['lcase_set'] = word_df['lcase'].map(lambda x: set(x))

In [15]:
word_df['n_unique_chars'] = word_df['lcase_set'].map(lambda x: len(x))

In [16]:
word_df = word_df.loc[(word_df['n_unique_chars'] == 5) & (word_df['n_letters'] == 5), :]
word_df = word_df.sort_values(by = 'lcase').reset_index(drop = True)

In [17]:
# using the word_group, select entries
word_df = word_df.drop_duplicates(subset = ['letters_sorted'])

In [18]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars
0,abdom,abdom,5,abdmo,"{a, m, b, o, d}",5
1,abend,abend,5,abden,"{a, b, n, e, d}",5
2,abets,abets,5,abest,"{a, b, t, e, s}",5
3,abhor,abhor,5,abhor,"{a, b, o, r, h}",5
4,abide,abide,5,abdei,"{a, b, i, e, d}",5


In [19]:
word_df.shape

(5977, 6)

In [20]:
# confirm that these words - a valid solution - are present
outcome_words = ['vibex', 'glyph', 'muntz', 'dwarf', 'jocks']
for ow in outcome_words:
    print(ow, ''.join(sorted(ow)), ''.join(sorted(ow)) in word_df['letters_sorted'].tolist())

vibex beivx True
glyph ghlpy True
muntz mntuz True
dwarf adfrw True
jocks cjkos True


In [21]:
word_df['word_id'] = range(0, word_df.shape[0])

In [22]:
word_df.shape

(5977, 7)

In [23]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id
0,abdom,abdom,5,abdmo,"{a, m, b, o, d}",5,0
1,abend,abend,5,abden,"{a, b, n, e, d}",5,1
2,abets,abets,5,abest,"{a, b, t, e, s}",5,2
3,abhor,abhor,5,abhor,"{a, b, o, r, h}",5,3
4,abide,abide,5,abdei,"{a, b, i, e, d}",5,4


In [24]:
word_id_list = word_df['word_id'].to_numpy(dtype = np.int16)

# BYTE ENCODE WORDS

In [25]:
word_df['word_byte'] = word_df['word'].map(byte_encode_words)

In [26]:
word_df.head()

,word,lcase,n_letters,letters_sorted,lcase_set,n_unique_chars,word_id,word_byte
0,abdom,abdom,5,abdmo,"{a, m, b, o, d}",5,0,20491
1,abend,abend,5,abden,"{a, b, n, e, d}",5,1,8219
2,abets,abets,5,abest,"{a, b, t, e, s}",5,2,786451
3,abhor,abhor,5,abhor,"{a, b, o, r, h}",5,3,147587
4,abide,abide,5,abdei,"{a, b, i, e, d}",5,4,283


In [27]:
word_byte_list = word_df['word_byte'].tolist()

## EXAMPLES OF BYTE COMPARISONS

In [28]:
w1 = 'abhor'
w2 = 'cleft'
w3 = 'frown'
w1b = byte_encode_words(w1)
w2b = byte_encode_words(w2)
w3b = byte_encode_words(w3)

In [29]:
# no letters in common
w1b & w2b

0

In [30]:
# letters in common
w1b & w3b

147456

In [31]:
w1b | w2b

673975

In [32]:
# this is the same as directly above
testo = byte_encode_words('abhorcleft')
testo

673975

In [33]:
lc_be = byte_encode_words(ascii_lowercase)

In [34]:
lc_be

67108863

In [35]:
word_byte_array = np.array(word_byte_list, dtype = np.int32)

In [36]:
word_byte_to_word_dict = {wb:lcase for wb, lcase in zip(word_df['word_byte'], word_df['lcase'])}

# BUILD LEVEL 2 USING COMBINATIONS

In [37]:
l2_df = build_l2(word_byte_list=word_byte_list)

(640023, 3)


In [38]:
l3_df = build_l3(word_byte_array = word_byte_array, l2_df=l2_df)

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
640000
(1272060, 5)


In [39]:
l4_df = build_l4(word_byte_array = word_byte_array, l3_df = l3_df)

0
10000
20000
30000
40000
50000
60000
70000
80000
90000
100000
110000
120000
130000
140000
150000
160000
170000
180000
190000
200000
210000
220000
230000
240000
250000
260000
270000
280000
290000
300000
310000
320000
330000
340000
350000
360000
370000
380000
390000
400000
410000
420000
430000
440000
450000
460000
470000
480000
490000
500000
510000
520000
530000
540000
550000
560000
570000
580000
590000
600000
610000
620000
630000
640000
650000
660000
670000
680000
690000
700000
710000
720000
730000
740000
750000
760000
770000
780000
790000
800000
810000
820000
830000
840000
850000
860000
870000
880000
890000
900000
910000
920000
930000
940000
950000
960000
970000
980000
990000
1000000
1010000
1020000
1030000
1040000
1050000
1060000
1070000
1080000
1090000
1100000
1110000
1120000
1130000
1140000
1150000
1160000
1170000
1180000
1190000
1200000
1210000
1220000
1230000
1240000
1250000
1260000
1270000
(54626, 7)


In [40]:
l5_df = build_l5(word_byte_array = word_byte_array, l4_df = l4_df)

0
10000
20000
30000
40000
50000
(11, 9)


In [ ]:
# try something else clever...

In [42]:
l2_df.shape

(640023, 3)

In [43]:
l2_array = l2_df['l2'].to_numpy()

# IDENTIFY ALL WORD GROUPS

In [41]:
l2_df_test = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)
l3_df_test = build_l3(word_byte_array = word_byte_array, l2_df = l2_df, focal_values=True)
l4_df_test = build_l4(word_byte_array = word_byte_array, l3_df = l3_df, focal_values=True)
l5_df_test = build_l5(word_byte_array = word_byte_array, l4_df = l4_df, focal_values=True)


NameError: name 'l2_set' is not defined

In [ ]:
l5_df.shape

In [ ]:
l5_df.head()

In [ ]:
l5_df.shape

In [ ]:
for idx in range(1, 3):
    cn = f"w{str(idx)}"
    ncn = f"w{str(idx)}lc"
    l2_df[ncn] = l2_df[cn].map(word_byte_to_word_dict)

In [ ]:
l2_df.head()

In [ ]:
l2_df.to_excel(excel_writer = 'l2_output.xlsx', index = False)

In [ ]:
for idx in range(1, 6):
    cn = f"w{str(idx)}b"
    ncn = f"w{str(idx)}"
    l5_df[ncn] = l5_df[cn].map(word_byte_to_word_dict)

In [ ]:
l5_df

In [ ]:
l5_df.to_excel(excel_writer='output_words.xlsx', index = False)

In [ ]:
# START AT LEVEL 2

In [ ]:
l2_set = set(l5_df['l2'].tolist())

In [ ]:
len(l2_set)

In [ ]:
l2_test.shape

In [ ]:
l2_df.shape

In [ ]:
l2_test.shape

In [ ]:
l2_df.to_csv(path_or_buf='l2.txt', sep = '\t', index = False)
l3_df.to_csv(path_or_buf='l3.txt', sep = '\t', index = False)
l4_df.to_csv(path_or_buf='l4.txt', sep = '\t', index = False)
l5_df.to_csv(path_or_buf='l5.txt', sep = '\t', index = False)

# LOAD IN PREVIOUS OUTPUT

In [ ]:
l2_df = pd.read_csv(filepath_or_buffer='l2.txt', sep = '\t')
l3_df = pd.read_csv(filepath_or_buffer='l3.txt', sep = '\t')
l4_df = pd.read_csv(filepath_or_buffer='l4.txt', sep = '\t')
l5_df = pd.read_csv(filepath_or_buffer='l5.txt', sep = '\t')

In [ ]:
l5_df.shape

In [ ]:
l2_set = set(l5_df['l2'].unique().tolist())

In [ ]:
l2_set

In [ ]:
test_l2_df = build_l2(word_byte_list=word_byte_list, focal_values=l2_set)

In [ ]:
test_l2_df.shape

In [ ]:
# get max valuesa
l2_df.max(axis = 0)

In [ ]:
l2_df['l2'].astype(np.int32).max()

In [ ]:
l3_df.head()

In [ ]:
l3_df.max(axis = 0)

In [ ]:
l4_df.max(axis = 0)

In [ ]:
my_columns = l5_df.columns.tolist()[:9]

In [ ]:
l5_df[my_columns].max(axis = 1)

In [ ]:
# join to get the different word combinations

In [ ]:
l5_df.head()

In [ ]:
l4_df.head()

In [ ]:
l4_df.shape

In [ ]:
l4_df.loc[l4_df['l4'] == 27784191, ]

In [ ]:
output_list = []
for ir5, row5 in l5_df.iterrows():
    l5 = row['l5']
    l5